# Per-Document PDF Processing with Ray Jobs on a Persistent RayCluster

This notebook submits **one RayJob per PDF document** to a persistent RayCluster,
simulating a real-time processing pattern where documents arrive individually.

## How it differs from batch processing

| Aspect | Batch (original) | Per-document (this notebook) |
|---|---|---|
| Job granularity | 1 job processes ALL PDFs | 1 job per PDF |
| Ray Data usage | Dataset + ActorPoolStrategy + repartition | None (single file, subprocess isolation only) |
| Cluster lifecycle | Created and torn down per batch | Persistent — stays up between jobs |
| Concurrency | Actors within a single job | Multiple independent jobs on the same cluster |
| Use case | Bulk ingestion | Real-time / on-demand processing |

## Architecture

```
Notebook (this file)
  │
  ├─ Connects to persistent RayCluster (already running)
  │
  ├─ For each PDF:
  │     submit_job(ray_single_doc_process.py, FILE_PATH=<path>)
  │       └─ Worker runs Docling in subprocess → writes MD + JSON to PVC
  │
  └─ Tracks all jobs: status, duration, results
```

## Prerequisites

- A **persistent RayCluster** already running (created by the batch notebook or manually)
- The **data-pvc** PVC mounted with input PDFs
- `ray_single_doc_process.py` in the notebook working directory

---
## Step 1 — Import Dependencies

In [ ]:
import os
import time

from codeflare_sdk import Cluster, ClusterConfiguration
from kubernetes import client as kclient
from kubernetes.client import (
    V1PersistentVolumeClaimVolumeSource,
    V1Volume,
    V1VolumeMount,
)
from kubernetes.client.rest import ApiException

---
## Step 2 — Connect to OpenShift

In [ ]:
# ---- REPLACE THESE WITH YOUR VALUES ----
TOKEN = "sha256~your-token-here"
API_URL = "https://api.your-cluster.example.com:443"
NAMESPACE = "your-namespace"

In [ ]:
!oc login --token={TOKEN} --server={API_URL}

---
## Step 3 — Configuration

Simplified parameters for per-document processing. No actor pool or repartitioning
needed — each job processes exactly one file.

In [ ]:
# ---------------------------------------------------------------------------
# Cluster (must match the existing persistent RayCluster)
# ---------------------------------------------------------------------------
CLUSTER_NAME = "ray-docling-processor"
IMAGE = "quay.io/cathaloconnor/docling-ray:latest"

# Cluster sizing (needed for ClusterConfiguration to connect)
NUM_WORKERS = 2
WORKER_CPUS = 4
WORKER_MEMORY_GB = 8
HEAD_CPUS = 4
HEAD_MEMORY_GB = 8

# ---------------------------------------------------------------------------
# Storage paths
# ---------------------------------------------------------------------------
PVC_NAME = "data-pvc"
PVC_MOUNT_PATH = "/mnt/data"
INPUT_PATH = "input/pdfs/10000"  # Relative to PVC_MOUNT_PATH
OUTPUT_PATH = "output-per-doc"   # Relative to PVC_MOUNT_PATH

# ---------------------------------------------------------------------------
# Per-document processing
# ---------------------------------------------------------------------------
CPUS_PER_ACTOR = 2       # CPUs for the processing subprocess
FILE_TIMEOUT = 600       # Per-file timeout in seconds
NUM_DOCS = 10            # How many docs to submit (for testing)
SUBMIT_DELAY = 0.5       # Seconds between submissions (simulates real-time arrival)

---
## Step 4 — Create or Connect to the RayCluster

If no cluster is running, this will create one. If it already exists, it
connects to it. After creation, we patch `rayStartParams` to set the correct
CPU scheduling (the CodeFlare SDK does not support this natively).

In [ ]:
# Volume mount (must match existing cluster)
shared_mount = V1VolumeMount(PVC_MOUNT_PATH, name="shared-data")
data_volume = V1Volume(
    name="shared-data",
    persistent_volume_claim=V1PersistentVolumeClaimVolumeSource(claim_name=PVC_NAME),
)

cluster_config = ClusterConfiguration(
    name=CLUSTER_NAME,
    namespace=NAMESPACE,
    head_cpu_requests=HEAD_CPUS,
    head_cpu_limits=HEAD_CPUS,
    head_memory_requests=HEAD_MEMORY_GB,
    head_memory_limits=HEAD_MEMORY_GB,
    num_workers=NUM_WORKERS,
    worker_cpu_requests=WORKER_CPUS,
    worker_cpu_limits=WORKER_CPUS,
    worker_memory_requests=WORKER_MEMORY_GB,
    worker_memory_limits=WORKER_MEMORY_GB,
    volume_mounts=[shared_mount],
    volumes=[data_volume],
    image=IMAGE,
    annotations={"odh.ray.io/secure-trusted-network": "false"},
    envs={
        "RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION": "0.1",
        "RAY_USE_TLS": "0",
    },
)

cluster = Cluster(config=cluster_config)
cluster.status()
print(f"\nDashboard: {cluster.cluster_dashboard_uri()}")

In [ ]:
# Create cluster if it doesn't exist, then connect
cluster.apply()
cluster.wait_ready(dashboard_check=False)

# Patch rayStartParams (not supported by CodeFlare SDK)
# - Workers: reserve 2 CPUs for Ray overhead, leaving WORKER_CPUS-2 schedulable
# - Head: 2 CPUs so the job driver can schedule there
import json as _json, subprocess as _sp

schedulable_cpus = WORKER_CPUS - 2
patch = [
    {
        "op": "add",
        "path": "/spec/workerGroupSpecs/0/rayStartParams/num-cpus",
        "value": str(schedulable_cpus),
    },
    {"op": "add", "path": "/spec/headGroupSpec/rayStartParams/num-cpus", "value": "2"},
]
_sp.run(
    ["oc", "patch", "raycluster", CLUSTER_NAME, "-n", NAMESPACE,
     "--type", "json", "-p", _json.dumps(patch)],
    check=True,
)
print(f"Patched: workers={schedulable_cpus} schedulable CPUs, head=2")

client = cluster.job_client
print(f"Connected to job server. Active jobs: {len(client.list_jobs())}")

---
## Step 5 — List Available PDFs

Discover PDFs on the PVC by listing files from a worker pod.

In [ ]:
import subprocess, json

# List PDFs on the PVC via a quick Ray job
list_job_id = client.submit_job(
    entrypoint=f"python -c \"import glob, json; print(json.dumps(sorted(glob.glob('{PVC_MOUNT_PATH}/{INPUT_PATH}/**/*.pdf', recursive=True))[:50]))\"",
    runtime_env={"env_vars": {}},
)

# Wait for it
while True:
    status = client.get_job_status(list_job_id)
    if status.is_terminal():
        break
    time.sleep(1)

logs = client.get_job_logs(list_job_id)
pdf_paths_on_pvc = json.loads(logs.strip().split("\n")[-1])
print(f"Found {len(pdf_paths_on_pvc)} PDFs on the PVC")
for p in pdf_paths_on_pvc[:10]:
    print(f"  {os.path.basename(p)}")
if len(pdf_paths_on_pvc) > 10:
    print(f"  ... and {len(pdf_paths_on_pvc) - 10} more")

---
## Step 6 — Submit Per-Document Jobs

Submit all jobs with a short delay between them, simulating documents arriving
in real time. Ray handles queuing, scheduling, and distribution across workers.

In [ ]:
# Verify the processing script exists
script_path = os.path.join(os.getcwd(), "ray_single_doc_process.py")
assert os.path.exists(script_path), f"Script not found: {script_path}"
print(f"Processing script: {script_path}")

In [ ]:
def submit_doc_job(file_path_on_pvc):
    """Submit a single-document processing job and return the submission ID."""
    fname = os.path.basename(file_path_on_pvc)
    # FILE_PATH is relative to PVC_MOUNT_PATH
    relative_path = os.path.relpath(file_path_on_pvc, PVC_MOUNT_PATH)

    job_id = client.submit_job(
        entrypoint="python ray_single_doc_process.py",
        entrypoint_num_cpus=CPUS_PER_ACTOR,  # Ray queues jobs until CPUs are free
        runtime_env={
            "working_dir": ".",
            "pip": ["opencv-python-headless", "pypdfium2", "orjson"],
            "env_vars": {
                "FILE_PATH": relative_path,
                "PVC_MOUNT_PATH": PVC_MOUNT_PATH,
                "OUTPUT_PATH": OUTPUT_PATH,
                "FILE_TIMEOUT": str(FILE_TIMEOUT),
                "CPUS_PER_ACTOR": str(CPUS_PER_ACTOR),
                "WRITE_JSON": "1",
                # Cache directories
                "HF_HOME": f"{PVC_MOUNT_PATH}/.cache/huggingface",
                "XDG_CACHE_HOME": f"{PVC_MOUNT_PATH}/.cache/xdg",
                "DOCLING_CACHE_DIR": f"{PVC_MOUNT_PATH}/.cache/docling",
            },
        },
    )
    return job_id, fname


# Select documents to process
docs_to_process = pdf_paths_on_pvc[:NUM_DOCS]
print(f"Will submit {len(docs_to_process)} documents with {SUBMIT_DELAY}s delay between each")
print(f"Each job requests {CPUS_PER_ACTOR} CPUs — Ray will queue excess jobs")
for i, p in enumerate(docs_to_process[:10]):
    print(f"  {i+1}. {os.path.basename(p)}")
if len(docs_to_process) > 10:
    print(f"  ... and {len(docs_to_process) - 10} more")

In [ ]:
# Phase 1: Submit all jobs with delay between each
pipeline_start = time.time()
all_jobs = {}  # job_id -> {fname, submit_time}

print(f"Submitting {len(docs_to_process)} jobs...")
print(f"=" * 60)

for i, pdf_path in enumerate(docs_to_process):
    job_id, fname = submit_doc_job(pdf_path)
    all_jobs[job_id] = {"fname": fname, "submit_time": time.time()}
    print(f"  [{i+1:3d}/{len(docs_to_process)}] SUBMITTED: {fname} -> {job_id}")
    if i < len(docs_to_process) - 1:
        time.sleep(SUBMIT_DELAY)

submit_duration = round(time.time() - pipeline_start, 1)
print(f"\nAll {len(all_jobs)} jobs submitted in {submit_duration}s")
print(f"Waiting for jobs to complete...")
print(f"=" * 60)

# Phase 2: Wait for all jobs to finish, printing as they complete
completed = []
pending = set(all_jobs.keys())

while pending:
    finished_this_round = []
    for job_id in list(pending):
        status = client.get_job_status(job_id)
        if status.is_terminal():
            info = all_jobs[job_id]
            duration = round(time.time() - info["submit_time"], 1)
            logs = client.get_job_logs(job_id)
            completed.append({
                "job_id": job_id,
                "fname": info["fname"],
                "status": str(status),
                "duration": duration,
                "logs": logs,
            })
            status_icon = "OK" if "SUCCEEDED" in str(status) else "FAIL"
            print(
                f"  [{len(completed):3d}/{len(all_jobs)}] {status_icon}: "
                f"{info['fname']} — {status} in {duration}s"
            )
            finished_this_round.append(job_id)

    for job_id in finished_this_round:
        pending.remove(job_id)

    if pending:
        time.sleep(5)

pipeline_duration = round(time.time() - pipeline_start, 1)
print(f"=" * 60)
print(f"All {len(completed)} jobs finished in {pipeline_duration}s (submit phase: {submit_duration}s)")

---
## Step 7 — Performance Report

Compare per-document job metrics to understand overhead and throughput.

In [ ]:
print("=" * 60)
print("PER-DOCUMENT PROCESSING REPORT")
print("=" * 60)

succeeded = [j for j in completed if "SUCCEEDED" in j["status"]]
failed = [j for j in completed if "SUCCEEDED" not in j["status"]]
durations = [j["duration"] for j in succeeded]

print(f"\nDocuments:       {len(completed)}")
print(f"Succeeded:       {len(succeeded)}")
print(f"Failed:          {len(failed)}")
print(f"Submit delay:    {SUBMIT_DELAY}s between jobs")
print(f"Submit phase:    {submit_duration}s")
print(f"Total wall clock: {pipeline_duration}s")

if durations:
    avg_dur = round(sum(durations) / len(durations), 1)
    min_dur = round(min(durations), 1)
    max_dur = round(max(durations), 1)
    median_dur = round(sorted(durations)[len(durations) // 2], 1)
    print(f"\nPer-job duration (submit to completion):")
    print(f"  Min:     {min_dur}s")
    print(f"  Max:     {max_dur}s")
    print(f"  Avg:     {avg_dur}s")
    print(f"  Median:  {median_dur}s")
    print(f"\nThroughput: {round(len(succeeded) / pipeline_duration, 2)} docs/sec")

    # Estimate overhead: if Docling alone takes ~10-30s, the rest is job overhead
    print(f"\n--- Timing Buckets ---")
    buckets = {"<30s": 0, "30-60s": 0, "60-120s": 0, "120-300s": 0, ">300s": 0}
    for d in durations:
        if d < 30: buckets["<30s"] += 1
        elif d < 60: buckets["30-60s"] += 1
        elif d < 120: buckets["60-120s"] += 1
        elif d < 300: buckets["120-300s"] += 1
        else: buckets[">300s"] += 1
    for bucket, count in buckets.items():
        bar = "#" * count
        print(f"  {bucket:>8s}: {count:3d} {bar}")

print(f"\n--- Per-Document Details ---")
for j in sorted(completed, key=lambda x: x["duration"]):
    icon = "OK" if "SUCCEEDED" in j["status"] else "FAIL"
    print(f"  [{icon}] {j['fname']:40s}  {j['duration']:>7.1f}s")

if failed:
    print(f"\n--- Failed Jobs ---")
    for j in failed:
        print(f"\n  {j['fname']} ({j['job_id']}):")
        log_lines = j["logs"].strip().split("\n")
        for line in log_lines[-5:]:
            print(f"    {line}")

print("=" * 60)

---
## Step 7b — Batch Baseline Comparison

Run the **same documents** as a single batch job using `ray_data_process.py` to
get a direct comparison. This uses Ray Data with ActorPoolStrategy — the same
approach as the batch notebook but submitted programmatically with the same files.

In [ ]:
# Verify batch script exists
batch_script = os.path.join(os.getcwd(), "ray_data_process.py")
assert os.path.exists(batch_script), f"Batch script not found: {batch_script}"

print(f"Running batch baseline with the same {NUM_DOCS} documents...")
batch_start = time.time()

batch_job_id = client.submit_job(
    entrypoint="python ray_data_process.py",
    entrypoint_num_cpus=CPUS_PER_ACTOR,
    runtime_env={
        "working_dir": ".",
        "pip": ["opencv-python-headless", "pypdfium2", "orjson"],
        "env_vars": {
            "PVC_MOUNT_PATH": PVC_MOUNT_PATH,
            "INPUT_PATH": INPUT_PATH,
            "OUTPUT_PATH": "output-batch-baseline",
            "NUM_FILES": str(NUM_DOCS),
            "MIN_ACTORS": "1",
            "MAX_ACTORS": "2",
            "CPUS_PER_ACTOR": str(CPUS_PER_ACTOR),
            "BATCH_SIZE": "4",
            "REPARTITION_FACTOR": "4",
            "FILE_TIMEOUT": str(FILE_TIMEOUT),
            "MAX_ERRORED_BLOCKS": "10",
            "RAY_DATA_ENABLE_RICH_PROGRESS_BARS": "true",
            "RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION": "0.1",
            "HF_HOME": f"{PVC_MOUNT_PATH}/.cache/huggingface",
            "XDG_CACHE_HOME": f"{PVC_MOUNT_PATH}/.cache/xdg",
            "DOCLING_CACHE_DIR": f"{PVC_MOUNT_PATH}/.cache/docling",
        },
    },
)
print(f"Batch job submitted: {batch_job_id}")

# Wait for completion
while True:
    status = client.get_job_status(batch_job_id)
    if status.is_terminal():
        break
    time.sleep(5)

batch_duration = round(time.time() - batch_start, 1)
batch_logs = client.get_job_logs(batch_job_id)
batch_status = str(status)

print(f"\nBatch job: {batch_status} in {batch_duration}s")
print("-" * 60)
print(batch_logs[-2000:])  # Last 2000 chars (performance report)

In [ ]:
# Side-by-side comparison
print("=" * 60)
print("COMPARISON: PER-DOCUMENT vs BATCH")
print("=" * 60)
print(f"{'':30s}  {'Per-Doc':>10s}  {'Batch':>10s}")
print(f"{'-'*30}  {'-'*10}  {'-'*10}")
print(f"{'Documents':30s}  {len(completed):>10d}  {NUM_DOCS:>10d}")
print(f"{'Total wall clock (s)':30s}  {pipeline_duration:>10.1f}  {batch_duration:>10.1f}")
if durations:
    per_doc_throughput = round(len(succeeded) / pipeline_duration, 2)
    batch_throughput = round(NUM_DOCS / batch_duration, 2) if batch_duration > 0 else 0
    print(f"{'Throughput (docs/sec)':30s}  {per_doc_throughput:>10.2f}  {batch_throughput:>10.2f}")
    print(f"{'Avg per-doc duration (s)':30s}  {avg_dur:>10.1f}  {'N/A':>10s}")
    speedup = round(pipeline_duration / batch_duration, 1) if batch_duration > 0 else 0
    print(f"\nBatch is {speedup}x {'faster' if speedup > 1 else 'slower'} than per-document")
    print(f"Per-doc overhead per job: ~{round(avg_dur - (batch_duration / NUM_DOCS), 1)}s")
print("=" * 60)

---
## Step 8 — Inspect Individual Job Logs

Use this cell to inspect the full logs for any specific job.

In [ ]:
# Print logs for a specific job (change index as needed)
job_index = 0  # 0 = first document
if completed:
    j = completed[job_index]
    print(f"Logs for: {j['fname']} ({j['job_id']})")
    print("-" * 60)
    print(j["logs"])

---
## Step 9 — Simulate Real-Time: Submit a Single Document On Demand

Use this cell to submit a single document interactively, simulating a user
uploading a file for processing.

In [ ]:
# Pick a specific document (change as needed)
single_doc = pdf_paths_on_pvc[0]
print(f"Submitting: {os.path.basename(single_doc)}")

t0 = time.time()
job_id, fname = submit_doc_job(single_doc)
print(f"Job ID: {job_id}")

# Wait for completion
while True:
    status = client.get_job_status(job_id)
    if status.is_terminal():
        break
    time.sleep(3)

duration = round(time.time() - t0, 1)
logs = client.get_job_logs(job_id)
print(f"\nStatus: {status} in {duration}s")
print("-" * 40)
print(logs)

---
## Step 10 — Cleanup

The cluster is persistent — leave it running for future submissions.
Only tear it down when you're done with all testing.

In [ ]:
# Delete the Ray cluster (only when done with all testing)
# cluster.down()